# Śledzenie obiektów

<img src="https://i.imgur.com/wKXXFkQ.png" width="500">

## Wstęp
W erze cyfrowej, w obliczu rosnącej lawinowo ilości danych wideo, zdolność do ich automatycznego rozpoznawania i interpretowania staje się kluczowa w wielu dziedzinach – od bezpieczeństwa publicznego po autonomiczne pojazdy. Technologie oparte na głębokim uczeniu rewolucjonizują sposób, w jaki przetwarzamy informacje wizualne. Kluczowym wyzwaniem jest tu detekcja i śledzenie obiektów na filmach wideo.

Celem tego zadania jest opracowanie algorytmu, który będzie w stanie analizować sekwencje ruchów w grze "trzy kubki". Uczestnicy mają za zadanie określić końcową pozycję kubków po serii ruchów, korzystając z analizy statycznych obrazów z każdej klatki nagrania.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

# Poniższe funkcje ułatwiają pracę z dostarczonymi danymi
# W kolejnych komórkach zobaczysz przykłady ich użycia
from utils.utils import get_level_info, get_video_data, display_video, download_and_replace_data

FINAL_EVALUATION_MODE = False
# W czasie sprawdzania Twojego rozwiązania, zmienimy tę wartość na True
# Wartość tej flagi M U S I zostać ustawiona na False w rozwiązaniu, które nam nadeślesz!

images, coordinates, _, path_to_images = get_video_data(level=3,video_id=0,dataset="example")
display_video(images,rescale=0.7,FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

## Zadanie 3: Zbuduj rozwiązanie od zera

W tym zadaniu podejdziesz do problemu identyfikacji obiektów na filmach wideo od podstaw. Poprzednie zadania wymagały użycia gotowych informacji o lokalizacji obiektów w klatkach. Tym razem będziesz musiał stworzyć algorytm, który będzie operować bezpośrednio na nieoznaczonych obrazach, co pozwoli na pełniejsze zrozumienie i opracowanie własnego systemu detekcji obiektów.

Napisz algorytm, który poradzi sobie ze zbiorem danych `level_3` bez podanych prostokątów ograniczających.

## Pliki zgłoszeniowe
Tylko ten notebook zawierający **kod** oraz **krótki raport** opisujący Twoje rozwiązanie (do 300 słów). Miejsce na raport znajdziesz na końcu tego notebooka.

## Ograniczenia
- Twoja funkcja powinna zwracać predykcje w maksymalnie 5 minut używając Google Colab bez GPU.

## Uwagi i wskazówki
- Testuj swoje rozwiązanie na zbiorze plików wideo `level_3`.
- **Skuteczność modelu**: przetestuj skuteczność modelu na zbiorze walidacyjnym używając dostarczonej przez nas funkcji **submission_script**, umieść ten wynik w raporcie.

## Ewaluacja
Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`. Za pomocą skryptu `validation_script.py` możesz upewnić się, że Twoje rozwiązanie zostanie prawidłowo wykonane na naszych serwerach oceniających.

Za to podzadanie możesz zdobyć pomiędzy 0 i 0.5 punktów. Zdobędziesz 0 punktów jeśli Twoje accuracy na zbiorze testowym będzie poniżej 30%. Jeśli będzie równe 100%, otrzymasz 0.5 punktu. Pomiędzy tymi wartościami, wynik rośnie liniowo z wartością metryki.

# Kod startowy

In [ ]:
# Poniższe biblioteki są wystarczające do wykonania wszystkich zadań
# Jeśli jednak chcesz użyć innych, sprawdź czy są dostępne na serwerze (requirements.txt)
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import IPython.display
import json
import PIL
import sklearn as sk
import gdown
import os

In [ ]:
# funkcja pomocnicza do ładowania danych
images, _, _, _ = get_video_data(level=3,video_id=0,dataset="example")

with open(os.path.join(os.getcwd(),'example_tracks','tracks_3_0.json'), 'r') as f:
    tracks = json.load(f)

for key in tracks.keys():
    tracks[key] = [tuple(el) for el in tracks[key]]

# funkcja pomocnicza do wyświetlania danych
display_video(images,
                tracks=tracks,
                rescale=0.7,
                FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

In [ ]:
# Pobieranie danych do podzadań 1, 2 i 3 (około ~646Mb), skrypt będzie wykonywał się parę minut
# Wystarczy, że pobierzesz dane tylko raz. Na serwerze sprawdzającym dane będą już pobrane,
# struktura plików będzie identyczna jak tutaj
if not FINAL_EVALUATION_MODE:
    download_and_replace_data()


In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# funkcja pomocnicza do testowania algorytmu
# Zwróć uwagę na to że funkcja ta działa inaczej niż w poprzednich podzadaniach
# algorytm przyjmuje jako argument listę obrazków, a nie koordynaty
def submission_script(algorithm,level,verbose=False,dataset="valid"):
    num_videos, _ = get_level_info(level=level,dataset=dataset)
    correct = []
    exception_messages = set()
    for video_number in range(num_videos):
        images, _, target, _ = get_video_data(level=level,video_id=video_number,dataset=dataset)
        try:
            prediction = algorithm(images)
            if tuple(target) == tuple(prediction):
                correct.append(1)
            else:
                correct.append(0)
            if verbose:
                print(f"Video: animation_{str(video_number).zfill(4)}")
                print(f"Prediction: {prediction}")
                print(f"Target:     {target}")
                print(f"Score: {tuple(target) == tuple(prediction)}", end='\n\n')
        except Exception as e:
            correct.append(0)
            exception_messages.add(str(e))
    if verbose:
        print(f"Accuracy: {np.mean(correct)}")
        print(f"Correctness: {correct}")
    return np.sum(correct) / num_videos, correct, exception_messages

# Twoje rozwiązanie

In [ ]:
import torchvision
from collections import deque

def image_to_tensor(img):
    return torch.tensor(np.array(img))[..., :3] / 255.0

def detect_edges(ts: torch.Tensor):
    edge_kernel_1 = torch.tensor([[[
        [1.0, 0.0, -1.0],
        [2.0, 0.0, -2.0],
        [1.0, 0.0, -1.0],
    ]]])
    edge_kernel_2 = torch.tensor([[[
        [-1.0, -1.0, -1.0],
        [-1.0,  8.0, -1.0],
        [-1.0, -1.0, -1.0],
    ]]])
    cv1 = torch.nn.functional.conv2d(ts, edge_kernel_1, padding="same")
    cv2 = torch.nn.functional.conv2d(ts, edge_kernel_2, padding="same")
    return cv1.gt(0.15) | cv2.abs().gt(0.1)

def detect_redness(red_channel, green_channel):
    return (red_channel - green_channel).gt(0.2) & (red_channel + green_channel).gt(0.5)

def detect_regions(ts: torch.Tensor):
    red_channel = ts[0].unsqueeze(dim=0)
    green_channel = ts[1].unsqueeze(dim=0)
    edges = detect_edges(red_channel)
    reds = detect_redness(red_channel, green_channel)
    regions = (reds & ~edges).permute(1, 2, 0)
    regions = regions.view((regions.shape[0], regions.shape[1]))
    return regions.to(dtype=torch.bool)

def get_average_of_regions(regions: torch.Tensor):
    regs = []
    
    W = regions.shape[0]
    H = regions.shape[1]
    regions = regions.numpy()

    queue = deque(range(W * H))

    for x in range(W):
        for y in range(H):
            if regions[x][y]:
                region_sum_x = 0
                region_sum_y = 0
                region_total = 0
                queue = [(x, y)]
                regions[x][y] = False
                while len(queue) > 0:
                    x, y = queue.pop()
                    region_sum_x += x
                    region_sum_y += y
                    region_total += 1
                    for nx, ny in [(1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1)]:
                        if regions[x + nx][y + ny]:
                            regions[x + nx][y + ny] = False
                            queue.append(((x + nx), (y + ny)))

                if region_total > 9:
                    regs.append([region_sum_y / region_total, region_sum_x / region_total])
    return regs

def get_error(avg1, avg2):
    return ((np.array(avg1) - np.array(avg2)) ** 2).mean()

def your_algorithm_task_3(images): # nie zmieniaj nazwy funkcji
    W = images[0].size[0]
    H = images[0].size[1]
    resize = torchvision.transforms.Resize((round(H / 3), round(W / 3)), torchvision.transforms.InterpolationMode.NEAREST)
    inreg = 0
    averages = []
    for img in images:
        ts = resize(image_to_tensor(img).permute(2, 0, 1))
        regions = detect_regions(ts)
        regs = get_average_of_regions(regions)
        if len(regs) == 3:
            averages.append(regs)
        else:
            inreg += 1

    
    crd1 = crd2 = crd3 = None
    for avg in averages:
        if crd1 == None:
            indexed_avg = [
                [avg[0][0], avg[0][1], 0],
                [avg[1][0], avg[1][1], 1],
                [avg[2][0], avg[2][1], 2],
            ]
            indexed_avg.sort()
            crd1 = avg[indexed_avg[0][2]]
            crd2 = avg[indexed_avg[1][2]]
            crd3 = avg[indexed_avg[2][2]]
        else:
            best_error = float('inf')
            best_perm = None
            for idx1, idx2, idx3 in [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]:
                err = get_error([crd1, crd2, crd3], [avg[idx1], avg[idx2], avg[idx3]])
                if err < best_error:
                    best_error = err
                    best_perm = [idx1, idx2, idx3]
            crd1 = avg[best_perm[0]]
            crd2 = avg[best_perm[1]]
            crd3 = avg[best_perm[2]]

    indexed_avg = [
        [crd1[0], crd1[1], 0],
        [crd2[0], crd2[1], 1],
        [crd3[0], crd3[1], 2],
    ]
    indexed_avg.sort()

    permutation = [indexed_avg[0][2], indexed_avg[1][2], indexed_avg[2][2]]
    return permutation

In [ ]:
# zapisz swój raport do zmiennej poniżej, abyśmy mogli go później automatycznie odczytać sprawdzaczką
raport_3 = \
"""
Raport z zadania:
Algorytm najpierw każdy obraz zmniejsza 3-krotnie aby zmniejszyć czas obliczeń. Następnie, korzystając z biblioteki pytorch, policzy dwie maski bitowe:
 - Pierwsza będzie ustawiona na 1 tylko na tych pikselach, dla których różnica między kanałem czerwonym a zielonym jest większa niż 0.2 oraz suma tych kanałów jest większa niż 0.5. W ten sposób zdobędziemy maskę która rozpozna, gdzie są kubki oraz weźmie tylko ich jaśniejszą część (ciemniejsza część może być problematyczna dla algorytmu, więc lepiej jest się jej pozbyć od razu)
 - Druga będzie wynikiem bitowego OR-a dwóch konwolucji na kanale czerwonym (chcemy wykryć krawędzie między dwoma kubkami jeśli na siebie nachodzą). Kernelami tych konwolucji są filtr Sobela oraz prosty wykrywacz krawędzi mający wszędzie wartości -1, ale +8 na środku.
Następnie algorytm od pierwszej maski "odejmuje" drugą (red_mask & ~edge_mask). Ta maska jedynie w około 2.5% klatek źle rozpoznaje krawędzie i łączy dwa różne kubki ze sobą. W pozostałych przypadkach zawiera ona 3 niespójne regiony. Wystarczy, że puścimy po nich algorytm BFS i policzymy średnie wartości każdego takiego regionu. Jeśli okaże się, że algorytm znalazł jest więcej lub mniej niż 3 regiony, po prostu odrzucamy tą klatkę. Pozostałe średnie pozycje kubków są użyte w algorytmie identycznym co w pierwszym podzadaniu aby uzyskać ostateczną permutację kubków.
Algorytm przetwarza jedną animację w około 3.8s i uzyskuje 100% skuteczności na wszystkich danych w zadaniu.
"""